# Ch 23 · Soft Bodies: Flex, Cloth, Rods and the Discrete Integrator

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ozzyozbourne/ai_lab/blob/master/dpdk-research/mujoco-warp-course/notebooks/23-soft-bodies-flex-discrete-integrator.ipynb)

Companion notebook for the chapter page `chapters/23-soft-bodies-flex-discrete-integrator.html` ([page](../chapters/23-soft-bodies-flex-discrete-integrator.html)). Pinned versions: mujoco-warp 4.0.1, warp-lang 1.19.0, mujoco-core 4.0.1.

**Before you start:** Runtime > Change runtime type > select a GPU. Then run the cells top to bottom.

In [ ]:
# Check the GPU Colab gave you (this fails harmlessly on a CPU runtime).
!nvidia-smi

In [ ]:
# Standard setup cell (identical in every notebook). Run it first.
# 1. Rendering backend must be chosen BEFORE mujoco is imported (rendering notebooks only).
import os
# (not a rendering notebook: MUJOCO_GL left unset)
# 2. Driver check: pip warp-lang 1.19.0 is built with CUDA 13 and needs an R580+ driver.
import shutil, subprocess
smi = ""
if shutil.which("nvidia-smi"):
    try:
        smi = subprocess.run(["nvidia-smi", "--query-gpu=driver_version,name", "--format=csv,noheader"],
                             capture_output=True, text=True).stdout.strip()
    except OSError:
        smi = ""
print(smi or "No NVIDIA GPU visible")
try:
    major = int(smi.split(".")[0]) if smi else 0
except ValueError:
    major = 0
# 3. Pick the Warp build: PyPI (CUDA 13) for R580+, else the +cu12 release wheel.
WARP = ("warp-lang==1.19.0" if major >= 580 else
        "https://github.com/NVIDIA/warp/releases/download/v1.19.0/"
        "warp_lang-1.19.0+cu12-py3-none-manylinux_2_28_x86_64.whl")
# 4. Install pinned versions ([viewer] pulls glfw + pyopengl + pillow via mujoco-core[rendering]).
!pip install -q "mujoco-warp[viewer]==4.0.1" "mujoco-core[rendering]==4.0.1" {WARP} mediapy matplotlib
# 5. Import, quiet the JIT log, verify that Warp itself sees the GPU (otherwise it silently uses CPU).
import importlib.metadata as md
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco
import mujoco_warp as mjw
wp.init()
ON_GPU = wp.get_cuda_device_count() > 0
if not ON_GPU:
    print("WARNING: Warp sees no GPU. Runtime > Change runtime type > GPU. Using small CPU budgets.")
print("device:", wp.get_device(), "| warp", wp.__version__, "| mujoco-core", md.version("mujoco-core"),
      "| mujoco-warp", md.version("mujoco-warp"))
GRAPH_COND = ON_GPU and wp.is_conditional_graph_supported()
print("conditional graphs supported:", GRAPH_COND)

Every code cell below is the same file as the matching block on the chapter page (`examples/23-soft-bodies-flex-discrete-integrator/`). Budgets grow on a GPU (`ON_GPU`) and stay small on a CPU. All outputs on the page are CPU outputs; timings printed here are **your** numbers, and CPU numbers are never GPU numbers. The examples run in the page's order, which is not their number order: 23.1, 23.2, 23.2b, 23.3, 23.8 (section 5), 23.4, 23.7 (section 6), 23.6 (section 7), 23.5 (section 8), 23.9.

In [ ]:
%%writefile my_env.py
"""my_env.py - the course's running helper module, snapshot after Chapter 14.

Ch 6 added make_worlds. Ch 8 added reset_done, nonfinite_reset, snapshot and restore.
Ch 9 added describe_overflow (Example 9.7).
Ch 10 added randomize (Example 10.9): in-place, graph-friendly friction re-randomization.
Ch 14 adds sensor_slices (Example 14.3): sensor name -> (adr, dim) in d.sensordata.
Later chapters extend this file (Ch 24 capture_step, Ch 26 obs/reward kernels,
Ch 30 the environments).
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

__all__ = ["make_worlds", "reset_done", "nonfinite_reset", "snapshot", "restore",
           "describe_overflow", "randomize", "sensor_slices"]


# --- Ch 6 ----------------------------------------------------------------------------------------
def make_worlds(xml, nworld, nconmax=None, njmax=None, batch_sizes=None, **sizes):
    """Compile `xml` and return (mjm, m, d) with `nworld` worlds (Ch 6 signature).

    nconmax / njmax are per world; batch_sizes (e.g. {"body_mass": nworld}) gives chosen Model
    fields their own rows (Ch 10); any other keyword (naconmax=, ...) goes to make_data.
    """
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm, batch_sizes=batch_sizes)
    d = mjw.make_data(mjm, nworld=nworld, nconmax=nconmax, njmax=njmax, **sizes)
    assert d.qpos.shape == (nworld, mjm.nq), d.qpos.shape
    return mjm, m, d


# --- Ch 8: resets --------------------------------------------------------------------------------
@wp.kernel
def _nonfinite_kernel(qpos_in: wp.array2d[float], qvel_in: wp.array2d[float],
                      done_out: wp.array[bool]):
    worldid = wp.tid()
    bad = bool(False)
    for i in range(qpos_in.shape[1]):
        if not wp.isfinite(qpos_in[worldid, i]):
            bad = True
    for i in range(qvel_in.shape[1]):
        if not wp.isfinite(qvel_in[worldid, i]):
            bad = True
    if bad:
        done_out[worldid] = True          # only ever sets flags; never clears another reason


def reset_done(m, d, done):
    """Reset the worlds whose done flag is True, then recompute derived quantities.

    `done` is a wp.array of bool with shape (nworld,). Everything stays on the device, so this
    can be captured in a CUDA graph together with mjw.step (Ch 24, Ch 26).
    """
    mjw.reset_data(m, d, reset=done)
    mjw.forward(m, d)                     # reset_data does not recompute xpos, sensors, ...


def nonfinite_reset(m, d, done):
    """OR `done` with "qpos or qvel has a NaN/Inf" (MJWarp has no mj_checkPos/mj_checkVel)."""
    wp.launch(_nonfinite_kernel, dim=d.nworld, inputs=[d.qpos, d.qvel], outputs=[done])


# --- Ch 8: state snapshots -----------------------------------------------------------------------
def snapshot(m, d, mjm, sig, buf=None):
    """Copy the state components in `sig` of every world into a (nworld, size) float array.

    Use State.PHYSICS for MPC (it includes HISTORY and PLASTIC; note it has no TIME bit),
    add State.WARMSTART for bitwise replays with contacts, State.INTEGRATION for everything.
    """
    size = mujoco.mj_stateSize(mjm, int(sig))
    if buf is None:
        buf = wp.zeros((d.nworld, size), dtype=float)
    # get_state does NOT check the size: an undersized buffer is written out of bounds.
    assert buf.shape == (d.nworld, size), f"state buffer {buf.shape} != {(d.nworld, size)}"
    mjw.get_state(m, d, buf, int(sig))
    return buf


def restore(m, d, mjm, buf, sig, active=None):
    """Write a snapshot back (only where the per-world bool mask `active` is True, if given)."""
    assert buf.shape == (d.nworld, mujoco.mj_stateSize(mjm, int(sig)))
    mjw.set_state(m, d, buf, int(sig), active)


# --- Ch 9: overflow ------------------------------------------------------------------------------
def describe_overflow(d, max_worlds=8):
    """Decode d.overflow: "none", or one line per flagged world plus a world count per bit.

    Loops over mjw.OverflowType (Python >= 3.11 iterates the single-bit members only), so bits
    added by a later MJWarp release are decoded without editing this function.
    """
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    lines = []
    flagged = np.nonzero(ov)[0]
    for w in flagged[:max_worlds]:
        names = [bit.name for bit in mjw.OverflowType if ov[w] & bit]
        lines.append(f"world {w}: " + " | ".join(names))
    if len(flagged) > max_worlds:
        lines.append(f"... and {len(flagged) - max_worlds} more flagged worlds")
    counts = [f"{bit.name} x{int(np.count_nonzero(ov & bit))}" for bit in mjw.OverflowType
              if (ov & bit).any()]
    lines.append("worlds per bit: " + ", ".join(counts))
    return "\n".join(lines)


# --- Ch 10: domain randomization ----------------------------------------------------------------
@wp.kernel
def _randomize_friction_kernel(
    seed: wp.array(dtype=int),          # device array: a graph replays with the CURRENT value
    episode: wp.array(dtype=int),       # per-world episode counter, bumped on every re-draw
    reset: wp.array(dtype=bool),        # which worlds to re-draw (all if every entry is True)
    lo: float,
    hi: float,
    geom_friction: wp.array2d(dtype=wp.vec3),   # m.geom_friction, shape (nworld, ngeom)
):
    worldid, geomid = wp.tid()
    if not reset[worldid]:
        return
    # one random stream per (seed, world, episode): reproducible and different every episode
    rng = wp.rand_init(seed[0], worldid * 7919 + episode[worldid] * 104729)
    mu = wp.randf(rng, lo, hi)
    f = geom_friction[worldid, geomid]
    geom_friction[worldid, geomid] = wp.vec3(mu, f[1], f[2])


@wp.kernel
def _bump_episode_kernel(reset: wp.array(dtype=bool), episode: wp.array(dtype=int)):
    worldid = wp.tid()
    if reset[worldid]:
        episode[worldid] = episode[worldid] + 1


def randomize(m, seed, episode, reset, lo=0.1, hi=1.0):
    """Draw a new sliding friction in [lo, hi) for every geom of the worlds marked in reset.

    Writes m.geom_friction in place (it must already have one row per world, e.g. from
    put_model(batch_sizes={"geom_friction": nworld})), so a captured CUDA graph that contains
    this call keeps working: only array CONTENTS change, never array objects.
    seed: wp.array(dtype=int, shape=(1,)); episode: wp.array(dtype=int, shape=(nworld,));
    reset: wp.array(dtype=bool, shape=(nworld,)).
    """
    nworld = reset.shape[0]
    assert m.geom_friction.shape[0] == nworld, "batch geom_friction to nworld first"
    wp.launch(_randomize_friction_kernel, dim=(nworld, m.ngeom),
              inputs=[seed, episode, reset, lo, hi], outputs=[m.geom_friction])
    wp.launch(_bump_episode_kernel, dim=nworld, inputs=[reset], outputs=[episode])


# --- Ch 14: sensors ------------------------------------------------------------------------------
def sensor_slices(mjm):
    """Return {sensor name: (adr, dim)} so d.sensordata[:, adr:adr + dim] is that sensor.

    Unnamed sensors get the key "sensor<i>". The addresses come from the compiled MjModel
    (mjm.sensor_adr, mjm.sensor_dim) and are the same in every world.
    """
    out = {}
    for i in range(mjm.nsensor):
        name = mujoco.mj_id2name(mjm, mujoco.mjtObj.mjOBJ_SENSOR, i) or f"sensor{i}"
        out[name] = (int(mjm.sensor_adr[i]), int(mjm.sensor_dim[i]))
    return out

## 1. What a flex is
A flex is vertices (one body with 3 slide joints each, unless pinned), edges and simplex elements (`dim` 1, 2, 3). `smooth.flex` fills `d.flexvert_xpos`, `d.flexedge_length` and `d.flexedge_J` per world. Example 23.1 loads the shipped `rope.xml`.

In [ ]:
# Example 23.1 - ex1_rope.py
"""Example 23.1 - a 1-D flex: two pinned ropes from the shipped test model rope.xml.

Loads mujoco_warp/test_data/flex/rope.xml (two 12-vertex ropes made with flexcomp dim="1",
vertex 0 of each pinned, edge equality constraints), prints the flex sizes and follows the
free tip of rope 1 for 100 steps in NWORLD worlds.
Run: python ex1_rope.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 100

ROPE_PATH = epath.resource_path("mujoco_warp") / "test_data/flex/rope.xml"
mjm = mujoco.MjModel.from_xml_path(str(ROPE_PATH))
print(f"nflex {mjm.nflex}  nflexvert {mjm.nflexvert}  nflexedge {mjm.nflexedge}  "
      f"nflexelem {mjm.nflexelem}  flex_dim {mjm.flex_dim}")
print(f"nbody {mjm.nbody}  nv {mjm.nv}  neq {mjm.neq}  "
      f"(22 unpinned vertices x 3 slide DOFs = {22 * 3})")
print("flex_vertadr", mjm.flex_vertadr, " flex_vertnum", mjm.flex_vertnum)

m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)          # io.py defaults: no contacts here (contype=0)
tip = int(mjm.flex_vertadr[0] + mjm.flex_vertnum[0] - 1)   # last vertex of rope 1
mjw.forward(m, d)                               # fills flexvert_xpos before the first step

for i in range(NSTEP + 1):
    if i % 20 == 0:
        p = d.flexvert_xpos.numpy()[0, tip]      # demo only: reads to host every 20 steps
        print(f"step {i:3d}  t={i * mjm.opt.timestep:.3f}s  rope-1 tip xyz {p}")
    if i < NSTEP:
        mjw.step(m, d)
wp.synchronize()

z = d.flexvert_xpos.numpy()[:, tip, 2]
print("tip z in every world:", z)
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

Example 23.2 reads the flex fields and the strain per edge. **Pitfall:** for `dim=2` you must set `elastic2d="stretch"`, `"bend"` or `"both"`; the default `"none"` gives no forces. The C compiler may print a WARNING about 'no equality constraints or passive forces' on stderr for this model; the membrane forces are applied anyway.

In [ ]:
# Example 23.2 - ex2_flex_fields.py
"""Example 23.2 - read flex Data: vertex positions, edge lengths and per-edge strain.

A 5x5 cloth (flexcomp type="grid" dim="2") hangs from two pinned corners. It has an elastic
membrane (<elasticity elastic2d="stretch" young=... thickness=...>), so its edges stretch under
gravity. After settling we compare d.flexedge_length with the rest lengths m.flexedge_length0.
Run: python ex2_flex_fields.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 400

HANGING_CLOTH = """
<mujoco model="hanging_cloth">
  <option timestep="0.002"/>
  <worldbody>
    <flexcomp name="cloth" type="grid" count="5 5 1" spacing=".05 .05 .05" pos="0 0 1"
              radius=".005" dim="2" mass=".05">
      <pin id="20 24"/>
      <elasticity young="5e4" poisson="0.2" thickness="1e-3" elastic2d="stretch"/>
      <contact contype="0" conaffinity="0"/>
    </flexcomp>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(HANGING_CLOTH)
print(f"nflexvert {mjm.nflexvert}  nflexedge {mjm.nflexedge}  nflexelem {mjm.nflexelem} "
      f"(triangles)  nv {mjm.nv}")
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=NWORLD)

print("Model  m.flexedge_length0  ", m.flexedge_length0.shape, m.flexedge_length0.dtype.__name__)
print("Model  m.flex_edge         ", m.flex_edge.shape, m.flex_edge.dtype.__name__)
print("Data   d.flexvert_xpos     ", d.flexvert_xpos.shape, d.flexvert_xpos.dtype.__name__)
print("Data   d.flexedge_length   ", d.flexedge_length.shape, d.flexedge_length.dtype.__name__)
print("Data   d.flexedge_velocity ", d.flexedge_velocity.shape)
print("Data   d.flexedge_J        ", d.flexedge_J.shape, "(sparse rows, m.flexedge_J_rownnz)")

for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

L0 = m.flexedge_length0.numpy()
L = d.flexedge_length.numpy()[0]
strain = (L - L0) / L0
edges = m.flex_edge.numpy()
order = np.argsort(strain)[::-1]
print(f"after {NSTEP} steps ({NSTEP * mjm.opt.timestep:.1f} s):")
print(f"  rest lengths: {np.unique(np.round(L0, 4))}  (grid edges and diagonals)")
print(f"  strain min {strain.min():+.4f}  mean {strain.mean():+.4f}  max {strain.max():+.4f}")
for e in order[:3]:
    print(f"  most stretched edge {e:2d} verts {edges[e]}  L0 {L0[e]:.4f}  L {L[e]:.4f}  "
          f"strain {strain[e]:+.4f}")
lowest = d.flexvert_xpos.numpy()[0, :, 2].min()
print(f"lowest vertex z {lowest:.4f}  (pinned corners stay at z=1)")
print("finite:", bool(np.isfinite(d.qpos.numpy()).all()))

## 2-3. Elasticity and rods
Edge-based stretch (`_flex_elasticity`), bending (`_flex_bending`), stable Neo-Hookean (DISCRETE only, MjSpec `elastic3d = 1`), plasticity (return mapping in `_advance`) and discrete elastic rods (`eval_rod_chains`, `rod_parallel_transport`). A `dim=1` flex with `<elasticity>` becomes a rod with bending and twist; it still needs an edge equality to keep its length, and a cantilever needs two pinned vertices.

In [ ]:
# Example 23.2b - ex2b_elastic_rod.py
"""Example 23.2b - an elastic rod: a dim=1 flex with <elasticity> becomes a discrete elastic rod.

A 1-D flexcomp with young/poisson gets bending and twist data from the compiler: one hinge per
interior vertex, and extra twist joints. We clamp vertices 0 and 1 to a body (one pinned vertex
would only be a hinge) and let the rod sag as a cantilever under the DISCRETE integrator, next
to the same chain without elasticity. Both keep their edge lengths with an edge equality:
for dim=1, <elasticity> adds bending and twist, not stretch.
Run: python ex2b_elastic_rod.py
"""
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 1
NSTEP = 250


def rod(elasticity):
    return f"""
<mujoco model="rod">
  <option integrator="discrete" timestep="0.002"/>
  <worldbody>
    <body name="clamp" pos="0 0 1">
      <flexcomp name="rod" type="grid" count="8 1 1" spacing=".05 .05 .05" dim="1"
                radius=".01" mass=".1">
        {elasticity}
        <pin id="0 1"/>
        <contact contype="0" conaffinity="0"/>
      </flexcomp>
    </body>
  </worldbody>
</mujoco>
"""


for name, el in (("elastic rod", '<elasticity young="1e7" poisson="0.3"/><edge equality="true"/>'),
                 ("rope (no elasticity)", '<edge equality="true"/>')):
    mjm = mujoco.MjModel.from_xml_string(rod(el))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD)
    mjw.forward(m, d)
    rest = d.flexvert_xpos.numpy()[0, -1].copy()
    for _ in range(NSTEP):
        mjw.step(m, d)
    wp.synchronize()
    tip = d.flexvert_xpos.numpy()[0, -1]
    print(f"{name}:")
    nfree = mjm.nflexvert - 2
    print(f"  nv {mjm.nv} = {nfree} free vertices x 3 + {mjm.nv - 3 * nfree} twist DOFs  "
          f"flex_bendingadr {mjm.flex_bendingadr}  nplastic {mjm.nplastic}")
    print(f"  rod hinges m.nflexrodhinge {m.nflexrodhinge}  rod edges m.nflexrodedge "
          f"{m.nflexrodedge}  m.flex_rod_has_twist {m.flex_rod_has_twist.numpy()}")
    print(f"  tip at rest {rest}, after {NSTEP * mjm.opt.timestep:.1f} s {tip}, "
          f"z drop {rest[2] - tip[2]:.4f} m")

## 5. Flex collision
`flex_collision` runs after the rigid narrowphase: flex AABBs, plane-vertex and element-geom candidates, de-duplication, then pool slots. Example 23.3 drops a cloth on a sphere and sizes the buffers.

In [ ]:
# Example 23.3 - ex3_cloth_drop.py
"""Example 23.3 - drop a 6x6 cloth on a sphere and size the contact and row buffers.

The cloth is a flexcomp type="grid" dim="2" (36 vertices, 50 triangles) whose edge lengths
are held by edge equality constraints. It starts 0.05 above a fixed sphere, so it touches
within the first 0.1 s, then drapes over it and the floor plane. We pass explicit nconmax and
njmax, step 300 times at dt = 0.002 and decode d.overflow (Ch 9).
Run: python ex3_cloth_drop.py [--dump-json PATH]
  --dump-json PATH writes 60 decimated frames of world 0's flexvert_xpos (cloth replay).
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 300

CLOTH_ON_SPHERE = """
<mujoco model="cloth_on_sphere">
  <option timestep="0.002" solver="CG" tolerance="1e-6"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="2 2 .1"/>
    <geom name="ball" type="sphere" size=".1" pos="0 0 .2"/>
    <flexcomp name="cloth" type="grid" count="6 6 1" spacing=".06 .06 .06" pos="0 0 .35"
              radius=".01" dim="2" mass=".1">
      <edge equality="true"/>
      <contact condim="3" selfcollide="none"/>
    </flexcomp>
  </worldbody>
</mujoco>
"""


def describe_overflow(d, max_worlds=8):
    """Same as my_env.describe_overflow (Ch 9): "none", or the flagged bits per world."""
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    flagged = np.nonzero(ov)[0]
    lines = [f"world {w}: " + " | ".join(b.name for b in mjw.OverflowType if ov[w] & b)
             for w in flagged[:max_worlds]]
    return "\n".join(lines)


mjm = mujoco.MjModel.from_xml_string(CLOTH_ON_SPHERE)
print(f"nflexvert {mjm.nflexvert}  nflexedge {mjm.nflexedge}  nflexelem {mjm.nflexelem}  "
      f"nv {mjm.nv}  neq {mjm.neq}")
m = mjw.put_model(mjm)
d0 = mjw.make_data(mjm, nworld=1)
print(f"io.py defaults for a flex model: nconmax {d0.naconmax}, njmax {d0.njmax} per world")

NCONMAX, NJMAX = 64, 384                  # measured below; the defaults (256, 512) also fit
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=NCONMAX, njmax=NJMAX)
sphere_top = 0.3

frames, peak_nacon, peak_nefc = [], 0, 0
for i in range(NSTEP):
    mjw.step(m, d)
    if i % 30 == 29 or i == 0:          # demo only: host reads every 30 steps
        nacon = int(d.nacon.numpy()[0])
        nefc = int(d.nefc.numpy().max())
        peak_nacon, peak_nefc = max(peak_nacon, nacon), max(peak_nefc, nefc)
        zmin = d.flexvert_xpos.numpy()[0, :, 2].min()
        print(f"step {i + 1:3d}  cloth min z {zmin:.4f}  nacon {nacon:3d} (all worlds)  "
              f"max nefc {nefc}")
    if args.dump_json and i % 5 == 4:
        frames.append(np.round(d.flexvert_xpos.numpy()[0].astype(np.float64), 3).tolist())
wp.synchronize()

xyz = d.flexvert_xpos.numpy()
r = np.linalg.norm(xyz[0] - np.array([0, 0, 0.2]), axis=1)
print(f"final: min z {xyz[:, :, 2].min():.4f}, closest vertex to sphere centre {r.min():.4f} "
      f"(radius 0.1 + cloth radius 0.01)")
print(f"centre vertex rests at z {xyz[0, :, 2].max():.4f}  (sphere top {sphere_top})")
print(f"peak nacon {peak_nacon} of naconmax {d.naconmax}; peak nefc {peak_nefc} of njmax {d.njmax}")
print("overflow:", describe_overflow(d))

if args.dump_json:
    with open(args.dump_json, "w") as f:
        elem = mjm.flex_elem[: 3 * mjm.nflexelem].reshape(-1, 3).tolist()   # triangles
        json.dump({"nworld": 1, "dt": 5 * float(mjm.opt.timestep), "kind": "mesh3d",
                   "elem": elem, "radius": 0.01, "sphere": [0, 0, 0.2, 0.1], "frames": frames},
                  f, separators=(",", ":"))
    print("wrote", args.dump_json, len(frames), "frames")

In [ ]:
import my_env                      # the running-project helper (Ch 9 describe_overflow)
print(my_env.describe_overflow(d))

In [ ]:
# Plot the cloth of Example 23.3 (world 0) after 300 steps, over the sphere.
xyz = d.flexvert_xpos.numpy()[0]
print("cloth vertices z range:", xyz[:, 2].min().round(4), "to", xyz[:, 2].max().round(4))
try:
    import matplotlib.pyplot as plt
    fig = plt.figure(figsize=(5, 4))
    ax = fig.add_subplot(projection="3d")
    elem = mjm.flex_elem[: 3 * mjm.nflexelem].reshape(-1, 3)
    ax.plot_trisurf(xyz[:, 0], xyz[:, 1], xyz[:, 2], triangles=elem, alpha=0.8)
    u, v = np.mgrid[0:2 * np.pi:20j, 0:np.pi:10j]
    ax.plot_wireframe(0.1 * np.cos(u) * np.sin(v), 0.1 * np.sin(u) * np.sin(v),
                      0.2 + 0.1 * np.cos(v), color="orange", linewidth=0.5)
    ax.set_box_aspect((1, 1, 0.7)); ax.set_title("Example 23.3: cloth on sphere")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

Flex contacts in the shared pool. Classify a contact side as flex by `contact.geom == -1`: in 4.0.1 rigid colliders do not reset `contact.flex/elem/vert`, so rigid contacts can carry stale flex ids.

In [ ]:
# Example 23.8 - ex8_flex_contacts.py
"""Example 23.8 - flex contacts next to rigid contacts in the one shared pool.

A 6x6 cloth (1 m wide) falls on scatter(30) (model M9: free spheres and capsules, Ch 15) over a
plane; no hfield and no SDF, which a flex model may not contain. We count the contacts of each
path, record the kernels that one mjw.collision call launches (wp.launch wrapped, as in
Example 18.9), and print the flex/elem/vert fields of the first flex contacts.
A contact side is a flex exactly when contact.geom is -1 on that side. In 4.0.1 the rigid
colliders do not reset contact.flex/elem/vert, so a rigid contact can carry stale flex ids
from an earlier step: classify by contact.geom, as support.contact_bodies_and_weights does.
Run: python ex8_flex_contacts.py [--dump-json PATH]
  --dump-json PATH writes the pool (worldid and path per slot) and the kernel list.
"""
import argparse
import collections
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from mujoco_warp._src.collision_driver import MJ_COLLISION_TABLE       # private API
from mujoco_warp._src.collision_driver import MJ_FLEX_COLLISION_TABLE  # private API

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 2
NSTEP = 1000 if ON_GPU else 250
G = mjw.GeomType


def scatter_bodies(n, seed=0, spacing=0.4, z0=0.3):
    """The bodies of scatter(n) (model M9, Ch 15)."""
    rng = np.random.default_rng(seed)
    per_side = 10
    out = []
    for i in range(n):
        layer, k = divmod(i, per_side * per_side)
        row, col = divmod(k, per_side)
        x = -0.5 * spacing * (per_side - 1) + spacing * col + rng.uniform(-0.02, 0.02)
        y = -0.5 * spacing * (per_side - 1) + spacing * row + rng.uniform(-0.02, 0.02)
        z = z0 + spacing * layer
        if i % 2 == 0:
            geom = f'<geom type="sphere" size="{rng.uniform(0.05, 0.1):.3f}"/>'
        else:
            e = rng.uniform(-180, 180, size=3)
            geom = (f'<geom type="capsule" size="{rng.uniform(0.05, 0.07):.3f} '
                    f'{rng.uniform(0.05, 0.1):.3f}" euler="{e[0]:.0f} {e[1]:.0f} {e[2]:.0f}"/>')
        out.append(f'<body name="b{i}" pos="{x:.3f} {y:.3f} {z:.3f}"><freejoint/>{geom}</body>')
    return "".join(out)


SCENE = f"""
<mujoco model="cloth_over_scatter">
  <option timestep="0.002" solver="CG" tolerance="1e-6"/>
  <worldbody>
    <light pos="0 0 4" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="3 3 .1"/>
    {scatter_bodies(30)}
    <flexcomp name="cloth" type="grid" count="6 6 1" spacing=".2 .2 .2" pos="0 -1.4 .5"
              radius=".01" dim="2" mass=".2">
      <edge equality="true"/>
      <contact condim="3" selfcollide="none"/>
    </flexcomp>
  </worldbody>
</mujoco>
"""

mjm = mujoco.MjModel.from_xml_string(SCENE)
m = mjw.put_model(mjm)
m.opt.warn_overflow = 0
print(f"ngeom {mjm.ngeom}  nflex {mjm.nflex}  nflexvert {mjm.nflexvert}  nv {mjm.nv}")
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=128, njmax=640)
for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

# ---- record the kernels of one collision call ----
calls = []
_launch = wp.launch


def recording_launch(kernel, dim, *a, **kw):
    calls.append((kernel.key.split("__")[0], np.atleast_1d(dim).tolist()))
    return _launch(kernel, dim, *a, **kw)


wp.launch = recording_launch
try:
    mjw.collision(m, d)
finally:
    wp.launch = _launch
wp.synchronize()
print(f"one mjw.collision call launches {len(calls)} kernels:")
for name, dim in calls:
    print(f"  {name:32s} dim {dim}")

# ---- contacts per path ----
nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
geom = d.contact.geom.numpy()[:nacon]
flex = d.contact.flex.numpy()[:nacon]
elem = d.contact.elem.numpy()[:nacon]
vert = d.contact.vert.numpy()[:nacon]
is_flex = (geom < 0).any(axis=1)          # a flex side has geom == -1
per_path = collections.Counter()
slot_kind = []
for g, flx in zip(geom, is_flex):
    if flx:
        gt = G(int(mjm.geom_type[g[g >= 0][0]]))
        key = f"flex-{gt.name} ({MJ_FLEX_COLLISION_TABLE[(gt, G.FLEX)].name})"
    else:
        t1, t2 = G(int(mjm.geom_type[g[0]])), G(int(mjm.geom_type[g[1]]))
        key = f"{t1.name}-{t2.name} ({MJ_COLLISION_TABLE[(t1, t2)].name})"
    per_path[key] += 1
    slot_kind.append(key)
print(f"after {NSTEP} steps, {NWORLD} worlds: nacon {nacon} of {d.naconmax}")
for k, v in sorted(per_path.items()):
    print(f"  {v:4d}  {k}")
stale = int(((flex >= 0).any(axis=1) & ~is_flex).sum())
print(f"rigid contacts whose (stale) contact.flex is not -1: {stale}")
print("first flex contact of each kind (geom, flex, elem, vert; -1 = not that kind of object):")
seen = set()
for c in np.nonzero(is_flex)[0]:
    kind = int(mjm.geom_type[geom[c][0]])
    if kind in seen:
        continue
    seen.add(kind)
    print(f"  contact {c}: geom {geom[c]}  flex {flex[c]}  elem {elem[c]}  vert {vert[c]}  "
          f"dist {d.contact.dist.numpy()[c]:+.4f}")
print(f"max nefc {int(d.nefc.numpy().max())} of njmax {d.njmax}; overflow:",
      "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"naconmax": d.naconmax, "nworld": NWORLD, "nstep": NSTEP,
                   "worldid": d.contact.worldid.numpy()[:nacon].tolist(), "kind": slot_kind,
                   "calls": calls, "per_path": dict(per_path)}, f, separators=(",", ":"))
    print("wrote", args.dump_json)

A volumetric flex: the shipped Floppy block (1242 tetrahedra, nv 1153) under Euler.

In [ ]:
# Example 23.4 - ex4_floppy_volumetric.py
"""Example 23.4 - a 3-D (volumetric) flex: the shipped Floppy model falls on a cylinder.

Loads mujoco_warp/test_data/flex/floppy.xml: a 24x4x4 vertex block made with flexcomp
dim="3" (tetrahedral elements) and <elasticity young="5e4" poisson="0.2">, above a hinged
cylinder. Prints the element counts and the block's centre height over time.
Run: python ex4_floppy_volumetric.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 1024 if ON_GPU else 1
NSTEP = 2000 if ON_GPU else 600

FLOPPY_PATH = epath.resource_path("mujoco_warp") / "test_data/flex/floppy.xml"
mjm = mujoco.MjModel.from_xml_path(str(FLOPPY_PATH))
f = 0
print(f"flex_dim {mjm.flex_dim[f]}  vertices {mjm.flex_vertnum[f]}  edges {mjm.flex_edgenum[f]}  "
      f"tetrahedra {mjm.flex_elemnum[f]}  nv {mjm.nv}")
print(f"integrator {mjw.IntegratorType(mjm.opt.integrator).name}  "
      f"solver {mjw.SolverType(mjm.opt.solver).name}  dt {mjm.opt.timestep}")
elem = mjm.flex_elem[mjm.flex_elemdataadr[f]: mjm.flex_elemdataadr[f] + 4 * mjm.flex_elemnum[f]]
print("first tetrahedron vertex ids:", elem[:4], "(dim + 1 = 4 per element)")

m = mjw.put_model(mjm)
print(f"nonzeros of the flex stiffness matrix K (m.nefmK, used by DISCRETE): {m.nefmK}")
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=256, njmax=1024)

mjw.step(m, d)                      # first call compiles the kernels
wp.synchronize()
t0 = time.perf_counter()
for i in range(1, NSTEP + 1):
    mjw.step(m, d)
    if i % 100 == 0:                # demo only: host reads every 100 steps
        x = d.flexvert_xpos.numpy()[0]
        print(f"t={(i + 1) * mjm.opt.timestep:.3f}s  centre z {x[:, 2].mean():.4f}  "
              f"min z {x[:, 2].min():.4f}  nacon {int(d.nacon.numpy()[0])}")
wp.synchronize()
dt = time.perf_counter() - t0
print("finite:", bool(np.isfinite(d.qpos.numpy()).all()),
      "| overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())
print(f"{NWORLD * NSTEP / dt:.0f} world-steps/s on {wp.get_device()} "
      + ("(GPU numbers)" if ON_GPU else "(CPU numbers are not GPU numbers)"))

## 6. Flex equalities
FLEX: one row per edge. FLEXSTRAIN: one row per stored deformation eigenmode of each interpolated cell (18 for one trilinear cell).

In [ ]:
# Example 23.7 - ex7_flexstrain.py
"""Example 23.7 - FLEXSTRAIN equality rows of an interpolated (trilinear) flex, in the efc strip.

Loads mujoco_warp/test_data/flex/moving_base_strain.xml: a 4x4x4 soft cube with dof="trilinear"
(its vertices follow 8 corner nodes) hangs under a box on a vertical slide joint and is held
in shape by <edge equality="strain"/>, which compiles to one FLEXSTRAIN equality per cell (this cube has a single cell).
We kick one corner node sideways, step, and print the equality types and where their rows sit
in d.efc (Ch 19 layout), plus how hard the strain rows push back.
Run: python ex7_flexstrain.py [--dump-json PATH]
"""
import argparse
import collections
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw
from etils import epath

np.set_printoptions(precision=4, suppress=True)

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 50
C = mujoco.mjtConstraint          # same values as MJWarp's types.ConstraintType

PATH = epath.resource_path("mujoco_warp") / "test_data/flex/moving_base_strain.xml"
mjm = mujoco.MjModel.from_xml_path(str(PATH))
print(f"nflex {mjm.nflex}  flex_interp {mjm.flex_interp} (1 = trilinear)  "
      f"cells {mjm.flex_cellnum[0]}  nodes {mjm.flex_nodenum[0]}  vertices {mjm.nflexvert}")
print(f"nv {mjm.nv} (slide 1 + 7 free nodes x 3: node 0 is pinned to the base)")
print(f"neq {mjm.neq}  eq_type {[mujoco.mjtEq(t).name for t in mjm.eq_type]}  "
      f"flex_edgeequality {mjm.flex_edgeequality} (3 = strain)")

m = mjw.put_model(mjm)
print(f"m.eq_flexstrain_adr {m.eq_flexstrain_adr.numpy()}  m.neq_flexstrain {m.neq_flexstrain}")
d = mjw.make_data(mjm, nworld=NWORLD, nconmax=16, njmax=64)
qvel = np.zeros((NWORLD, mjm.nv), dtype=np.float32)
qvel[:, -3] = 0.5                         # last free node: 0.5 m/s along x (a shear kick)
d.qvel.assign(qvel)
for _ in range(NSTEP):
    mjw.step(m, d)
wp.synchronize()

nefc = d.nefc.numpy()
types = d.efc.type.numpy()[0, : nefc[0]]
ids = d.efc.id.numpy()[0, : nefc[0]]
print(f"nefc per world {nefc}  (njmax {d.njmax})")
runs = []
for t, i in zip(types, ids):
    if runs and runs[-1][0] == t and runs[-1][1] == i:
        runs[-1][2] += 1
    else:
        runs.append([t, i, 1])
for t, i, n in runs:
    print(f"  {n:3d} rows  {C(t).name[8:]:16s} id {i}")
pos = d.efc.pos.numpy()[0, : nefc[0]]
force = d.efc.force.numpy()[0, : nefc[0]]
eq = types == C.mjCNSTR_EQUALITY
print(f"strain rows: |pos| max {np.abs(pos[eq]).max():.2e}  "
      f"|force| max {np.abs(force[eq]).max():.4f}  "
      f"({eq.sum()} = 3 x 8 node DOFs - 6 rigid-body modes)")
print(f"base slide qpos {d.qpos.numpy()[0, 0]:.4f}  cube lowest vertex z "
      f"{d.flexvert_xpos.numpy()[0, :, 2].min():.4f}")
print("overflow:", "none" if not d.overflow.numpy().any() else d.overflow.numpy())

if args.dump_json:
    rows = [[C(t).name[8:], int(i), int(n)] for t, i, n in runs]
    with open(args.dump_json, "w") as f:
        json.dump({"njmax": d.njmax, "nefc": int(nefc[0]), "runs": rows}, f)
    print("wrote", args.dump_json)

## 7. The DISCRETE integrator
`(M + hD + h^2 K) a = f - hKv`: `eff_build` -> `eff_shift` -> `eff_actuation` + `regularize_constraint` -> `eff_solve` (PCG, `capture_while` on GPU) -> the constraint solve in the same metric. Example 23.6: a stiff cloth that Euler cannot step.

In [ ]:
# Example 23.6 - ex6_discrete_integrator.py
"""Example 23.6 - a stiff cloth: Euler blows up, DISCRETE stays finite at the same timestep.

The 5x5 hanging cloth of Example 23.2, with a 10x stiffer membrane (young 5e5). Explicit
(semi-implicit) Euler is only stable while the stiffest mode satisfies omega * dt < 2; here it
does not. DISCRETE builds an effective metric M_hat = M + h D + h^2 K from the flex stiffness
(derivative.eff_build) and solves the step in that metric, so it stays stable. IMPLICITFAST
refuses flex elasticity (forward.py::check_discrete). We step both integrators in the same
script and watch for non-finite values (the non-finite guard of Ch 8, without the reset).
Run: python ex6_discrete_integrator.py
"""
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

np.set_printoptions(precision=4, suppress=True)

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 2
NSTEP = 400


def hanging_cloth(young, integrator):
    return f"""
<mujoco model="stiff_cloth">
  <option timestep="0.002" integrator="{integrator}" solver="CG" tolerance="1e-8"/>
  <worldbody>
    <flexcomp name="cloth" type="grid" count="5 5 1" spacing=".05 .05 .05" pos="0 0 1"
              radius=".005" dim="2" mass=".05">
      <pin id="20 24"/>
      <elasticity young="{young}" poisson="0.2" thickness="1e-3" elastic2d="stretch"/>
      <contact contype="0" conaffinity="0"/>
    </flexcomp>
  </worldbody>
</mujoco>
"""


@wp.kernel
def nonfinite_flag(qpos: wp.array2d(dtype=float), bad: wp.array(dtype=int)):
    w, j = wp.tid()
    if not wp.isfinite(qpos[w, j]):
        bad[w] = 1


def run(young, integrator):
    mjm = mujoco.MjModel.from_xml_string(hanging_cloth(young, integrator))
    m = mjw.put_model(mjm)
    d = mjw.make_data(mjm, nworld=NWORLD)
    bad = wp.zeros(NWORLD, dtype=int)
    first_bad = None
    mjw.step(m, d)                                # compile
    wp.synchronize()
    t0 = time.perf_counter()
    for i in range(1, NSTEP):
        mjw.step(m, d)
        wp.launch(nonfinite_flag, dim=(NWORLD, mjm.nq), inputs=[d.qpos, bad])
        if first_bad is None and i % 10 == 0 and bad.numpy().any():   # demo: host read
            first_bad = i
    wp.synchronize()
    sec = time.perf_counter() - t0
    L0 = m.flexedge_length0.numpy()
    strain = (d.flexedge_length.numpy() - L0) / L0
    finite = bool(np.isfinite(d.qpos.numpy()).all())
    a = np.abs(strain)
    smax = float(np.nanmax(a)) if np.isfinite(a).any() else float("nan")
    return m, finite, first_bad, smax, d.solver_niter.numpy(), sec


print(f"{'young':>7} {'integrator':>11} {'finite':>6} {'first NaN step':>14} "
      f"{'max |strain|':>12} {'ms/step (' + ('GPU' if ON_GPU else 'CPU') + ')':>13}")
for young in (5e4, 5e5):
    for integ in ("Euler", "discrete"):
        m, finite, first_bad, smax, niter, sec = run(young, integ)
        fb = "-" if first_bad is None else f"<= {first_bad}"
        sm = f"{smax:.4f}" if finite else "nan/inf"
        print(f"{young:7.0e} {integ:>11} {str(finite):>6} {fb:>14} {sm:>12} "
              f"{1000 * sec / (NSTEP - 1):13.2f}")
        if integ == "discrete":
            print(f"        DISCRETE: K has m.nefmK = {m.nefmK} nonzeros; "
                  f"solver_niter in the last step {niter}")

mjm = mujoco.MjModel.from_xml_string(hanging_cloth(5e5, "Euler"))
mjm.opt.integrator = mujoco.mjtIntegrator.mjINT_IMPLICITFAST   # the C compiler refuses it too
try:
    m = mjw.put_model(mjm)
    mjw.forward(m, mjw.make_data(mjm, nworld=1))
except ValueError as e:
    print("implicitfast:", e)

## 8. What 4.0.1 refuses
Every unsupported flex combination, with the exception and message.

In [ ]:
# Example 23.5 - ex5_unsupported_combos.py
"""Example 23.5 - every flex combination that MJWarp 4.0.1 refuses, and the message it gives.

Each case builds a small model, then calls mjw.put_model (the io.py checks) and, if that
passes, mjw.forward (forward.py::check_discrete runs at the start of every forward/step).
We catch the exception and print its type and message. The MuJoCo 4.0.1 C compiler itself
refuses four integrator combinations (SNH or passive flex contact without DISCRETE, flex
elasticity with implicitfast, edge stiffness with DISCRETE), so those cases compile with an
accepted integrator and then set mjm.opt.integrator, to reach MJWarp's own checks. SNH is
MjSpec-only (it cannot be written to MJCF): that case sets spec.flexes[0].elastic3d = 1.
Run: python ex5_unsupported_combos.py [--dump-json PATH]
"""
import argparse
import json

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()


def cloth(extra="", inner="", count="4 4 1", dim=2, attrs=""):
    """A small flexcomp grid; `inner` goes inside <flexcomp>, `extra` into <worldbody>."""
    return f"""<mujoco><worldbody>{extra}
      <flexcomp name="f" type="grid" count="{count}" spacing=".1 .1 .1" dim="{dim}"
                radius=".01" mass=".1" {attrs}>{inner}</flexcomp>
    </worldbody></mujoco>"""


def with_option(xml, option):
    return xml.replace("<mujoco>", f"<mujoco>{option}", 1)


EDGE_EQ = '<edge equality="true"/>'
STRETCH = '<elasticity young="1e4" poisson="0.2" thickness="1e-3" elastic2d="stretch"/>'
SDF_ASSET = """<extension><plugin plugin="mujoco.sdf.torus"><instance name="torus">
  <config key="radius1" value="0.3"/><config key="radius2" value="0.1"/></instance></plugin>
  </extension><asset><mesh name="torus"><plugin instance="torus"/></mesh></asset>"""
HFIELD_ASSET = '<asset><hfield name="h" nrow="3" ncol="3" size="1 1 .1 .1"/></asset>'


DISCRETE = '<option integrator="discrete"/>'


def snh_model():
    xml = cloth(inner='<elasticity young="1e4" poisson="0.3"/>', count="3 3 3", dim=3)
    spec = mujoco.MjSpec.from_string(with_option(xml, DISCRETE))
    spec.flexes[0].elastic3d = 1          # stable Neo-Hookean; the C compiler needs DISCRETE...
    mjm = spec.compile()
    mjm.opt.integrator = mujoco.mjtIntegrator.mjINT_EULER   # ...so switch afterwards
    return mjm


def switched(xml, integrator):
    """Compile xml as given, then change the integrator on the compiled MjModel."""
    def build():
        mjm = mujoco.MjModel.from_xml_string(xml)
        mjm.opt.integrator = integrator
        return mjm
    return build


CASES = [
    # (key, what, MJCF or builder)
    ("snh", "stable Neo-Hookean, integrator Euler", snh_model),
    ("quadratic", 'dof="quadratic" interpolation',
     cloth(inner=STRETCH, count="3 3 3", dim=3, attrs='dof="quadratic"')),
    ("eq_sleep", "flex edge equality + sleep",
     with_option(cloth(inner=EDGE_EQ), '<option><flag sleep="enable"/></option>')),
    ("hfield", "flex + hfield geom (contype=0)",
     cloth(extra='<geom type="hfield" hfield="h" contype="0" conaffinity="0"/>')
     .replace("<worldbody>", HFIELD_ASSET + "<worldbody>", 1)),
    ("sdf", "flex + SDF geom",
     cloth(extra='<geom type="sdf" mesh="torus" pos="0 0 -1"><plugin instance="torus"/></geom>')
     .replace("<worldbody>", SDF_ASSET + "<worldbody>", 1)),
    ("rigid", 'rigid flex (flexcomp rigid="true")', cloth(attrs='rigid="true"')),
    ("passive_interp", "passive contact, trilinear flex",
     with_option(cloth(inner='<contact passive="true" selfcollide="none"/>', count="3 3 3", dim=3,
                       attrs='dof="trilinear"'), DISCRETE)),
    ("self_interp", "self-collision, trilinear flex",
     cloth(inner='<contact selfcollide="auto"/>', count="3 3 3", dim=3, attrs='dof="trilinear"')),
    ("noslip", "discrete + noslip_iterations",
     with_option(cloth(inner=EDGE_EQ), '<option integrator="discrete" noslip_iterations="2"/>')),
    ("attach_newton", "discrete + Newton, cloth on a moving body",
     with_option(f"""<mujoco><worldbody><body name="base" pos="0 0 1"><freejoint/>
       <geom type="box" size=".05 .05 .05"/>
       <flexcomp name="f" type="grid" count="3 3 1" spacing=".1 .1 .1" dim="2" radius=".01"
                 mass=".1"><pin id="0"/>{STRETCH}</flexcomp></body></worldbody></mujoco>""",
                 '<option integrator="discrete" solver="Newton"/>')),
    ("implicitfast", "implicitfast + flex elasticity",
     switched(cloth(inner=STRETCH), mujoco.mjtIntegrator.mjINT_IMPLICITFAST)),
    ("passive_euler", "passive contact, integrator Euler",
     switched(with_option(cloth(inner='<contact passive="true"/>'), DISCRETE),
              mujoco.mjtIntegrator.mjINT_EULER)),
    ("edge_stiff", "discrete + edge stiffness (1-D rope)",
     switched(cloth(inner='<edge stiffness="100"/>', count="5 1 1", dim=1),
              mujoco.mjtIntegrator.mjINT_DISCRETE)),
]

results = []
for key, what, src in CASES:
    mjm = src() if callable(src) else mujoco.MjModel.from_xml_string(src)
    stage = "put_model"
    try:
        m = mjw.put_model(mjm)
        stage = "forward"
        d = mjw.make_data(mjm, nworld=1)
        mjw.forward(m, d)
        stage, err, msg = "-", "OK", "accepted"
    except Exception as e:                      # NotImplementedError or ValueError
        err, msg = type(e).__name__, str(e)
    results.append({"key": key, "what": what, "stage": stage, "error": err, "message": msg})
    print(f"[{key}] {what}\n    {stage}: {err}: {msg}")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump(results, f, indent=0)
    print("wrote", args.dump_json, len(results), "cases")

## 9. Rendering a cloth frame
The Ch 27 batch renderer puts a 2-D flex into its own BVH and offsets the faces by +-radius (`refit_flex_bvh`, `flex_render_smooth`). This works on CPU at a tiny resolution and is fast on a GPU.

In [ ]:
# Batch-render one cloth frame (Ch 27 renderer; runs on CPU at a small resolution, fast on GPU)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

ON_GPU = wp.get_device().is_cuda
RES = 256 if ON_GPU else 48
SCENE = """<mujoco><option timestep="0.002" solver="CG" tolerance="1e-6"/><worldbody>
  <light pos="0 0 3" dir="0 0 -1" directional="true"/>
  <camera name="cam" pos="0 -0.9 0.6" xyaxes="1 0 0 0 0.55 0.83"/>
  <geom type="plane" size="2 2 .1" rgba=".85 .85 .85 1"/>
  <geom name="ball" type="sphere" size=".1" pos="0 0 .2" rgba="1 .5 .1 1"/>
  <flexcomp name="cloth" type="grid" count="6 6 1" spacing=".06 .06 .06" pos="0 0 .35"
            radius=".01" dim="2" mass=".1" rgba=".2 .5 .9 1">
    <edge equality="true"/><contact condim="3" selfcollide="none"/>
  </flexcomp></worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=1, nconmax=64, njmax=384)
for _ in range(150):
    mjw.step(m, d)
rc = mjw.create_render_context(mjm, nworld=1, cam_res=(RES, RES), render_rgb=True,
                               render_depth=False, render_seg=True, flex_render_smooth=True)
mjw.refit_bvh(m, d, rc)
mjw.render(m, d, rc)
seg = rc.seg_data.numpy()[0]
print("pixels showing the flex (objtype 9 = mjOBJ_FLEX):", int((seg[:, 1] == 9).sum()), "of", RES * RES)
rgb = wp.zeros((1, RES, RES), dtype=wp.vec3)
mjw.get_rgb(rc, 0, rgb)
try:
    import matplotlib
    import matplotlib.pyplot as plt
    plt.imshow(np.clip(rgb.numpy()[0], 0, 1)); plt.axis("off"); plt.title("cloth after 0.3 s")
    plt.show()
except ImportError:
    print("skipped: matplotlib not installed")

## 10. Part V checkpoint: cloth on a box stack under a throughput budget
Size the buffers (the flex candidate buffer needs room too), then sweep integrator x solver x cone and pick the fastest config with no overflow bit (ITERATIONS included).

In [ ]:
# Example 23.9 - ex9_checkpoint_cloth_stack.py
"""Example 23.9 - Part V checkpoint "Cloth on a box stack under a throughput budget".

Model M11 cloth(6): a plane, box_stack(2) (model M4) and a 6x6 flexcomp cloth started 0.05
above the top box; no hfield and no SDF. Edge equality keeps the cloth inextensible, so both
IMPLICITFAST and DISCRETE are allowed (check_discrete only refuses flex elasticity under the
implicit integrators).
1. Size nconmax/njmax with the Ch 9 recipe: a rollout with generous buffers, the per-world
   peak contacts and rows, +25%, rounded up to the ladder. Then check: flex collision first
   writes candidates (before de-duplication) into a buffer that is also naconmax long, so a
   flex scene can set NARROWPHASE although nacon stayed below naconmax. Double what overflowed.
2. Sweep integrator x solver x cone with those sizes. For each config report the mean and max
   solver_niter, the overflow bits (ITERATIONS = some world hit the iteration cap), a
   stability check and world-steps/s. Pick the fastest config that converges in every world
   without overflow.
Run: python ex9_checkpoint_cloth_stack.py [--dump-json PATH]
CPU numbers are tiny and only rank the configs on this machine; on Colab the GPU prints yours.
"""
import argparse
import itertools
import json
import math
import time

import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

parser = argparse.ArgumentParser()
parser.add_argument("--dump-json", default=None)
args, _ = parser.parse_known_args()

ON_GPU = wp.get_device().is_cuda
NWORLD = 4096 if ON_GPU else 4
NSTEP = 1000 if ON_GPU else 150
LABEL = "(GPU numbers)" if ON_GPU else "(CPU dev box, not GPU numbers)"
ITERATIONS, TOLERANCE = 50, 1e-6
LADDER = [(2 + k % 2) * 2 ** (k // 2 + 3) for k in range(19)]   # io.py size ladder (Ch 9)


def box_stack_bodies(n=2, size=0.1):
    return "".join(f'<body name="box{i}" pos="0 0 {size + 2 * size * i + 0.001 * i:.4f}">'
                   f'<freejoint/><geom type="box" size="{size} {size} {size}"/></body>'
                   for i in range(n))


def cloth(n=6, z=0.45, integrator="implicitfast", solver="Newton", cone="pyramidal"):
    """Model M11: plane + box_stack(2) + an n x n cloth whose lowest point starts at z."""
    return f"""
<mujoco model="cloth_on_box_stack">
  <option timestep="0.005" integrator="{integrator}" solver="{solver}" cone="{cone}"
          iterations="{ITERATIONS}" tolerance="{TOLERANCE}"/>
  <worldbody>
    <light pos="0 0 3" dir="0 0 -1"/>
    <geom name="floor" type="plane" size="5 5 .1"/>
    {box_stack_bodies(2)}
    <flexcomp name="cloth" type="grid" count="{n} {n} 1" spacing=".06 .06 .06"
              pos="0 0 {z}" radius=".01" dim="2" mass=".1">
      <edge equality="true"/>
      <contact condim="3" selfcollide="none"/>
    </flexcomp>
  </worldbody>
</mujoco>
"""


def round_up_ladder(x):
    return next((v for v in LADDER if v >= x), int(x))


def describe_overflow(d):
    ov = d.overflow.numpy()
    if not ov.any():
        return "none"
    return ", ".join(f"{b.name} x{int(np.count_nonzero(ov & b))}" for b in mjw.OverflowType
                     if (ov & b).any())


def rollout(xml, nconmax, njmax, measure=False):
    mjm = mujoco.MjModel.from_xml_string(xml)
    m = mjw.put_model(mjm)
    m.opt.warn_overflow = 0
    d = mjw.make_data(mjm, nworld=NWORLD, nconmax=nconmax, njmax=njmax)
    mjw.step(m, d)                                   # compile outside the timed loop
    wp.synchronize()
    peak_con = np.zeros(NWORLD, int)
    peak_efc = np.zeros(NWORLD, int)
    niters = []
    nsamp = NSTEP // 10                              # timed runs: sample solver_niter every 10 steps
    niter_buf = wp.zeros(nsamp * NWORLD, dtype=int)  # on the device, copied out after the clock stops
    t0 = time.perf_counter()
    for i in range(NSTEP):
        mjw.step(m, d)
        if measure:                                  # sizing run: host reads every step (not timed)
            niters.append(d.solver_niter.numpy().copy())
            nacon = min(int(d.nacon.numpy()[0]), d.naconmax)
            w = d.contact.worldid.numpy()[:nacon]
            peak_con = np.maximum(peak_con, np.bincount(w, minlength=NWORLD))
            peak_efc = np.maximum(peak_efc, d.nefc.numpy())
        elif i % 10 == 9:                            # device-to-device copy, no host sync
            wp.copy(niter_buf, d.solver_niter, dest_offset=(i // 10) * NWORLD, count=NWORLD)
    wp.synchronize()
    sec = time.perf_counter() - t0
    niters = np.array(niters) if measure else niter_buf.numpy().reshape(nsamp, NWORLD)
    zbox = d.xpos.numpy()[:, 2, 2]                   # top box centre z, every world
    ok = bool(np.isfinite(d.qpos.numpy()).all()) and bool((np.abs(zbox - 0.301) < 0.02).all())
    bits = {b.name for b in mjw.OverflowType if (d.overflow.numpy() & b).any()}
    return dict(sec=sec, niter_mean=float(niters.mean()), niter_max=int(niters.max()),
                bits=bits, overflow=describe_overflow(d), stable=ok,
                peak_con=peak_con, peak_efc=peak_efc, mjm=mjm)


# ---- 1. size the buffers (Ch 9 recipe) ----
probe = rollout(cloth(), nconmax=256, njmax=512, measure=True)
mjm = probe["mjm"]
print(f"M11 cloth(6): nv {mjm.nv}  nflexvert {mjm.nflexvert}  nflexedge {mjm.nflexedge}  "
      f"ngeom {mjm.ngeom}")
print(f"sizing rollout ({NSTEP} steps, defaults 256/512): peak contacts per world "
      f"{probe['peak_con'].max()}, peak nefc {probe['peak_efc'].max()}")
NCONMAX = round_up_ladder(math.ceil(1.25 * probe["peak_con"].max()))
NJMAX = round_up_ladder(math.ceil(1.25 * probe["peak_efc"].max()))
print(f"recipe: nconmax={NCONMAX}, njmax={NJMAX} (peak x 1.25, ladder)")
FIX = {"NARROWPHASE": "nconmax", "BROADPHASE": "nconmax", "NEFC": "njmax"}
for attempt in range(4):
    r = rollout(cloth(), NCONMAX, NJMAX)
    bits = r["bits"] & set(FIX)
    print(f"check {attempt}: nconmax={NCONMAX} njmax={NJMAX} -> overflow {r['overflow']}")
    if not bits:
        break
    if "nconmax" in {FIX[b] for b in bits}:
        NCONMAX *= 2
    if "njmax" in {FIX[b] for b in bits}:
        NJMAX *= 2

# ---- 2. sweep the solver knobs ----
rows = []
print(f"{'integrator':12s} {'solver':6s} {'cone':9s} {'niter mean/max':>14s} "
      f"{'stable':>6s} {'world-steps/s':>13s}  overflow")
for integ, solver, cone in itertools.product(("implicitfast", "discrete"), ("Newton", "CG"),
                                             ("pyramidal", "elliptic")):
    r = rollout(cloth(integrator=integ, solver=solver, cone=cone), NCONMAX, NJMAX)
    r["wsps"] = NWORLD * NSTEP / r["sec"]
    rows.append(dict(integrator=integ, solver=solver, cone=cone,
                     niter_mean=round(r["niter_mean"], 2), niter_max=r["niter_max"],
                     stable=r["stable"],
                     overflow=r["overflow"], wsps=round(r["wsps"], 1)))
    print(f"{integ:12s} {solver:6s} {cone:9s} {r['niter_mean']:8.1f}/{r['niter_max']:<5d} "
          f"{str(r['stable']):>6s} {r['wsps']:13.0f}  {r['overflow']}")

good = [r for r in rows if r["stable"] and r["overflow"] == "none"]
best = max(good, key=lambda r: r["wsps"]) if good else None
if best:
    print(f"choice: integrator={best['integrator']} solver={best['solver']} cone={best['cone']} "
          f"nconmax={NCONMAX} njmax={NJMAX} -> {best['wsps']:.0f} world-steps/s "
          f"on {wp.get_device()} {LABEL}")
else:
    print("no config passed: raise iterations or the buffers and rerun")

if args.dump_json:
    with open(args.dump_json, "w") as f:
        json.dump({"device": str(wp.get_device()), "nworld": NWORLD, "nstep": NSTEP,
                   "nconmax": NCONMAX, "njmax": NJMAX, "rows": rows}, f, indent=0)
    print("wrote", args.dump_json)

## Part V review
Answer for yourself, then check against the chapter page (the quiz there gives instant feedback).
1. In which order does `make_constraint` write row blocks? *(equality, friction, limit, contact; flex rows are in the equality block)*
2. What minimizes the per-world Gauss cost in MJWarp? *(CG or Newton with an exact linesearch; no PGS)*
3. Which solver does sleep require? *(Newton; flex equalities are refused with sleep)*
4. Which `DeterminismType` bit changes kernels in 4.0.1? *(ATOMICS; the CONTACTS and CONSTRAINT sorts are stubs)*
5. How does DISCRETE change a row's `D`? *(divided by (sum J^2/qH)/(sum J^2/M) in `regularize_constraint`)*
6. A cloth model with an hfield floor fails at `put_model`. Fix? *(replace the hfield by a plane, boxes or meshes)*

## Exercise 23.1: build a rope and check its sizes
Write the MJCF and the DOF count.

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

N = 10
ROPE = ...  # your MJCF: <body name="anchor" ...><flexcomp ... dim="1">...</flexcomp></body>
mjm = mujoco.MjModel.from_xml_string(ROPE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2)
for _ in range(50):
    mjw.step(m, d)
assert mjm.nflexvert == N and mjm.nflexedge == N - 1
assert mjm.nv == ...            # how many DOFs?
assert d.flexvert_xpos.shape == (2, N)
assert np.abs(d.flexedge_length.numpy() - m.flexedge_length0.numpy()).max() < 1e-3
print("Exercise 23.1 passed")

In [ ]:
#@title Exercise 23.1: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

N = 10
ROPE = f"""<mujoco><option timestep="0.001" solver="CG"/><worldbody>
  <body name="anchor" pos="0 0 1">
    <flexcomp name="rope" type="grid" count="{N} 1 1" spacing=".05 .05 .05" radius=".01"
              dim="1" mass=".1">
      <edge equality="true"/><contact contype="0" conaffinity="0"/><pin id="0"/>
    </flexcomp>
  </body></worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(ROPE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2)
for _ in range(50):
    mjw.step(m, d)
assert mjm.nflexvert == N and mjm.nflexedge == N - 1
assert mjm.nv == 3 * (N - 1)    # the pinned vertex has no DOFs
assert d.flexvert_xpos.shape == (2, N)
assert np.abs(d.flexedge_length.numpy() - m.flexedge_length0.numpy()).max() < 1e-3
print("Exercise 23.1 passed")

## Exercise 23.2: the cloth must not sink into the sphere

In [ ]:
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

CENTRE, R, RADIUS, TOL = np.array([0, 0, 0.2]), 0.1, 0.01, 0.005
SCENE = """<mujoco><option timestep="0.002" solver="CG" tolerance="1e-6"/><worldbody>
  <geom type="plane" size="2 2 .1"/>
  <geom name="ball" type="sphere" size=".1" pos="0 0 .2"/>
  <flexcomp name="cloth" type="grid" count="6 6 1" spacing=".06 .06 .06" pos="0 0 .35"
            radius=".01" dim="2" mass=".1">
    <edge equality="true"/><contact condim="3" selfcollide="none"/>
  </flexcomp></worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=64, njmax=384)
for _ in range(300):
    mjw.step(m, d)
xyz = d.flexvert_xpos.numpy()                         # (nworld, nflexvert, 3)
dist = ...                                            # distance of every vertex to CENTRE
assert dist.min() >= R + RADIUS - TOL
assert ...                                            # top vertex per world >= 0.3 - TOL
assert not d.overflow.numpy().any()
print("Exercise 23.2 passed")

In [ ]:
#@title Exercise 23.2: solution (double-click to show the code)
import mujoco
import numpy as np
import warp as wp
wp.config.log_level = wp.LOG_WARNING
import mujoco_warp as mjw

CENTRE, R, RADIUS, TOL = np.array([0, 0, 0.2]), 0.1, 0.01, 0.005
SCENE = """<mujoco><option timestep="0.002" solver="CG" tolerance="1e-6"/><worldbody>
  <geom type="plane" size="2 2 .1"/>
  <geom name="ball" type="sphere" size=".1" pos="0 0 .2"/>
  <flexcomp name="cloth" type="grid" count="6 6 1" spacing=".06 .06 .06" pos="0 0 .35"
            radius=".01" dim="2" mass=".1">
    <edge equality="true"/><contact condim="3" selfcollide="none"/>
  </flexcomp></worldbody></mujoco>"""
mjm = mujoco.MjModel.from_xml_string(SCENE)
m = mjw.put_model(mjm)
d = mjw.make_data(mjm, nworld=2, nconmax=64, njmax=384)
for _ in range(300):
    mjw.step(m, d)
xyz = d.flexvert_xpos.numpy()                         # (nworld, nflexvert, 3)
dist = np.linalg.norm(xyz - CENTRE, axis=-1)
assert dist.min() >= R + RADIUS - TOL, dist.min()     # no vertex sinks into the sphere
assert xyz[:, :, 2].max(axis=1).min() >= 0.3 - TOL    # the cloth still rests on the sphere top
assert not d.overflow.numpy().any()
print("Exercise 23.2 passed", round(float(dist.min()), 4))

## Stretch: cloth with and without self-collision
Drop a 12x12 cloth (spacing 0.04) on the sphere with `selfcollide="none"` and `"auto"`. Count self-contacts (both `geom` entries -1 and `flex[0] == flex[1]`), compare peak `nacon`, overflow bits (raise `nconmax`), and time per step. Record the extra kernels of one `mjw.collision` call as in Example 23.8. On a GPU, scale `nworld` from 1 to 4096.

In [ ]:
# your self-collision experiment here